# 04 — Clinical Validation & Fairness
Validate model performance and check fairness across demographic groups.

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.utils.data_loader import load_sample_data
from src.utils.preprocessor import Preprocessor
from src.models.explainable_model import ExplainableMedicalAI
from src.compliance.fairness_checker import FairnessChecker
from src.compliance.audit_logger import AuditLogger
from src.compliance.regulatory_reports import RegulatoryReportGenerator

In [ ]:
(X_train, X_test, y_train, y_test), feature_names = load_sample_data()
prep = Preprocessor(feature_names)
X_train_s = prep.fit_transform(X_train)
X_test_s  = prep.transform(X_test)

model = ExplainableMedicalAI(model_type='decision_tree')
model.train(X_train_s, y_train, feature_names)
metrics = model.evaluate(X_test_s, y_test)

print(f"Accuracy : {metrics['accuracy']}")
print(f"ROC-AUC  : {metrics['roc_auc']}")

In [ ]:
# Build test DataFrame for group analysis
df_test = pd.DataFrame(X_test, columns=feature_names)
df_test['y_true']  = y_test
df_test['y_pred']  = model.predict(X_test_s)
df_test['y_proba'] = model.predict_proba(X_test_s)

# Create age groups
df_test['age_group'] = pd.cut(
    df_test['age'], bins=[0, 50, 65, 120],
    labels=['<50', '50-65', '>65']
).astype(str)

# Sex labels
df_test['sex_label'] = df_test['sex'].map({0: 'Female', 1: 'Male'})

df_test.head()

In [ ]:
# ── Fairness Analysis ─────────────────────────────────────────────
fc = FairnessChecker()

fairness_report = fc.full_report(
    y_true  = df_test['y_true'].values,
    y_pred  = df_test['y_pred'].values,
    y_proba = df_test['y_proba'].values,
    groups_dict = {
        'sex':       df_test['sex_label'].values,
        'age_group': df_test['age_group'].values,
    }
)

print(f"Overall fairness PASS: {fairness_report['_summary']['overall_fairness_pass']}")
for attr in ['sex', 'age_group']:
    r = fairness_report[attr]
    print(f"\n{attr.upper()}")
    print(f"  Demographic Parity Δ : {r['demographic_parity_difference']}  {'✓' if r['dp_pass'] else '✗'}")
    print(f"  Equalized Odds (TPR) : {r['equalized_odds']['tpr_difference']}  {'✓' if r['eo_pass'] else '✗'}")
    for g, gm in r['group_performance'].items():
        print(f"    {g:10s}  Acc={gm['accuracy']}  Sens={gm['sensitivity']}  Spec={gm['specificity']}")

In [ ]:
# ── Fairness chart ────────────────────────────────────────────────
sex_gp = fairness_report['sex']['group_performance']
groups = list(sex_gp.keys())
sens   = [sex_gp[g]['sensitivity'] for g in groups]
spec   = [sex_gp[g]['specificity']  for g in groups]

x = np.arange(len(groups))
fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(x - 0.2, sens, 0.35, label='Sensitivity', color='#63B3ED')
ax.bar(x + 0.2, spec, 0.35, label='Specificity',  color='#4FD1C5')
ax.set_xticks(x); ax.set_xticklabels(groups)
ax.set_ylim(0, 1); ax.set_ylabel('Rate')
ax.set_title('Fairness Monitor — Performance by Sex')
ax.legend(); plt.tight_layout()
plt.savefig('../logs/fairness_sex.png', dpi=120)
plt.show()

In [ ]:
# ── Regulatory Report ─────────────────────────────────────────────
audit   = AuditLogger()
reg_gen = RegulatoryReportGenerator()

reg_report = reg_gen.generate(
    audit_summary   = audit.compliance_summary(),
    fairness_report = fairness_report,
    model_metrics   = {**metrics, 'n_test': len(y_test)},
)

md = reg_gen.to_markdown(reg_report, output_path='../docs/compliance_report_generated.md')
print('Regulatory report written to docs/compliance_report_generated.md')